In [1]:
import pandas as pd

In [3]:
df = pd.read_excel("FinGraph_transactions (1).xlsx")
print(df)

   transaction_id sender_account receiver_account  amount         timestamp  \
0            T001           A101             B201     500  2026-08-11 10:01   
1            T002           A102             B202    1200  2026-08-11 10:02   
2            T003           A103             B203     750  2026-08-11 10:03   
3            T004           A104             B204    2300  2026-08-11 10:04   
4            T005           A105             B205     950  2026-08-11 10:05   
..            ...            ...              ...     ...               ...   
60           T061           A120             B220     800  2026-08-11 11:20   
61           T062           A121             B221    1500  2026-08-11 11:22   
62           T063           A122             B222     600  2026-08-11 11:24   
63           T064           A123             B223    2100  2026-08-11 11:26   
64           T065           A124             B224     950  2026-08-11 11:28   

      sender_ip  
0   192.168.1.1  
1   192.168.1.2

In [4]:
print(df.head())

  transaction_id sender_account receiver_account  amount         timestamp  \
0           T001           A101             B201     500  2026-08-11 10:01   
1           T002           A102             B202    1200  2026-08-11 10:02   
2           T003           A103             B203     750  2026-08-11 10:03   
3           T004           A104             B204    2300  2026-08-11 10:04   
4           T005           A105             B205     950  2026-08-11 10:05   

     sender_ip  
0  192.168.1.1  
1  192.168.1.2  
2  192.168.1.3  
3  192.168.1.4  
4  192.168.1.5  


In [5]:
print(df.shape)

(65, 6)


In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    65 non-null     object
 1   sender_account    65 non-null     object
 2   receiver_account  65 non-null     object
 3   amount            65 non-null     int64 
 4   timestamp         65 non-null     object
 5   sender_ip         65 non-null     object
dtypes: int64(1), object(5)
memory usage: 3.2+ KB
None


In [8]:
print(df.isnull().sum())

transaction_id      0
sender_account      0
receiver_account    0
amount              0
timestamp           0
sender_ip           0
dtype: int64


In [9]:
starburst = df.groupby("receiver_account")["sender_account"].nunique().sort_values(ascending=False)

print(starburst)

receiver_account
SHELL01    50
B201        1
B202        1
B203        1
B204        1
B205        1
B206        1
B207        1
B208        1
B209        1
B210        1
B220        1
B221        1
B222        1
B223        1
B224        1
Name: sender_account, dtype: int64


In [10]:
threshold = 10

suspicious_receivers = starburst[starburst >= threshold]

print(suspicious_receivers)

receiver_account
SHELL01    50
Name: sender_account, dtype: int64


In [11]:
risk_score = min(starburst["SHELL01"] * 2, 100)

print(risk_score)

100


In [13]:
suspicious_accounts = df[
    df["receiver_account"] == "SHELL01"
]["sender_account"].unique()

print(suspicious_accounts)

['A501' 'A502' 'A503' 'A504' 'A505' 'A506' 'A507' 'A508' 'A509' 'A510'
 'A511' 'A512' 'A513' 'A514' 'A515' 'A516' 'A517' 'A518' 'A519' 'A520'
 'A521' 'A522' 'A523' 'A524' 'A525' 'A526' 'A527' 'A528' 'A529' 'A530'
 'A531' 'A532' 'A533' 'A534' 'A535' 'A536' 'A537' 'A538' 'A539' 'A540'
 'A541' 'A542' 'A543' 'A544' 'A545' 'A546' 'A547' 'A548' 'A549' 'A550']


In [14]:
shell_transactions = df[df["receiver_account"] == "SHELL01"]

total_amount = shell_transactions["amount"].sum()
average_amount = shell_transactions["amount"].mean()

print("Total amount:", total_amount)
print("Average transaction:", average_amount)

Total amount: 492000
Average transaction: 9840.0


In [15]:
alert = {
    "receiver_account": "SHELL01",
    "pattern": "STARBURST",
    "unique_senders": len(suspicious_accounts),
    "total_amount": total_amount,
    "average_transaction": average_amount,
    "risk_score": risk_score,
    "status": "HIGH RISK"
}

alert

{'receiver_account': 'SHELL01',
 'pattern': 'STARBURST',
 'unique_senders': 50,
 'total_amount': np.int64(492000),
 'average_transaction': np.float64(9840.0),
 'risk_score': np.int64(100),
 'status': 'HIGH RISK'}

In [16]:
alerts_df = pd.DataFrame([alert])

print(alerts_df)

  receiver_account    pattern  unique_senders  total_amount  \
0          SHELL01  STARBURST              50        492000   

   average_transaction  risk_score     status  
0               9840.0         100  HIGH RISK  


In [17]:
alerts_df.to_excel("FinGraph_Fraud_Alerts.xlsx", index=False)
print(alerts_df)

  receiver_account    pattern  unique_senders  total_amount  \
0          SHELL01  STARBURST              50        492000   

   average_transaction  risk_score     status  
0               9840.0         100  HIGH RISK  
